# Seeded shard-pick eval — manual verification

Companion to `connito/shared/eval_shard_pick.py`. Walks through every property of the new eval-data path so you can eyeball it before flipping `DataCfg.eval_source_seeded_shard_pick=True` for production.

## What this notebook checks

1. **Pick is deterministic from seed** — same `int_seed` ⇒ same `(shard, offset)` for each source.
2. **In-shard offset stays inside the offset bound** — `hash % offset_bound` always yields a safe value; with module-load validation, `offset + min_headroom_rows ≤ actual_shard_rows` always.
3. **Per-source independence** — two configured sources at the same seed pick independently (different hash domain).
4. **Cross-seed coverage** — across many seeds, picks distribute roughly uniformly over the shard space.
5. **Live shard read** — actually pulls one chosen shard from HF, skips by the chosen offset, and prints the first few rows. This is the path the validator will take in production.
6. **Equivalence with the canonical load path** — verifies that `data_files=[shard]` yields rows from inside the dataset (not garbled or schema-shifted output).

## Design (path B — safe floor)

Instead of enumerating every shard's exact row count (which for C4-en json.gz would mean downloading and counting all 1024 shards, ~hours of work), the policy registry declares either:

- **`row_count_source="constant"`** (C4-en): use a `safe_floor_rows` constant for the mod, validated at module load against a small spot-check sample (`verified_shard_rows`). Coverage loss per shard = `(actual_rows − safe_floor) / actual_rows` ≈ 4.6% for C4. Module-load validation guarantees `safe_floor + min_headroom_rows ≤ min(verified)` so the read window can't run short.

- **`row_count_source="parquet_footer"`** (Nemotron): read the exact `num_rows` from the parquet footer at pick time (one ~32 KB HTTP-range read). Bound becomes `actual_rows − min_headroom_rows`.

## Threat-model recap

The new path retains today's anti-memorization story: `combined_seed` mixes the late-bound `MinerCommit2` last-block hash, so a miner can't precompute the pick before the block is sealed. The shard-pick scheme widens the *coverage* (every shard reachable across seeds) without changing the per-round overfit budget (~340 k rows per source reachable per round for C4).

If you run this against a production validator host, set `MY_INT_SEED` below to a real `combined_seed`-derived value to see exactly what that round would pick.

In [ ]:
from __future__ import annotations

import sys
from collections import Counter
from pathlib import Path

# Resolve repo root regardless of whether the kernel was started
# from repo root or from the notebook's directory.
CWD = Path.cwd()
for candidate in (CWD, *CWD.parents):
    if (candidate / "connito" / "shared" / "dataloader.py").exists():
        sys.path.insert(0, str(candidate))
        break

from connito.shared import eval_shard_pick
print("loaded", eval_shard_pick.__file__)

## (1) Determinism — same seed yields the same pick

Two calls with the same `int_seed` against the same source should give byte-identical `(shard_path, in_shard_offset)`. Without this, two validators with the same `combined_seed` would draw different batches → weight consensus breaks for the round.

In [ ]:
MY_INT_SEED = 0xdeadbeef  # change to a real combined_seed-derived int to inspect a specific round

SOURCES = [
    ("allenai/c4", "en"),
    # Nemotron is gated — uncomment if your HF token has access:
    # ("nvidia/Nemotron-CC-Math-v1", "4plus"),
]

for repo_id, name in SOURCES:
    a = eval_shard_pick.pick_shard_for_source(repo_id=repo_id, name=name, int_seed=MY_INT_SEED)
    b = eval_shard_pick.pick_shard_for_source(repo_id=repo_id, name=name, int_seed=MY_INT_SEED)
    assert (a.shard_path, a.in_shard_offset) == (b.shard_path, b.in_shard_offset), \
        f"Non-deterministic pick for {repo_id}"
    print(f"{repo_id:35s} shard={a.shard_path:50s}"
          f" bound={a.offset_bound:>8d} rows={a.shard_rows:>8d} offset={a.in_shard_offset:>7d}")

## (2) Mod safety — offset always inside the bound

Run 256 seeds through the picker and confirm `0 ≤ offset < offset_bound` for every result. The bound is:

- `safe_floor_rows` for constant-source policies (C4), pre-validated at module load against `verified_shard_rows`.
- `actual_rows − min_headroom_rows` for parquet-footer policies (Nemotron), computed at pick time.

If a future change breaks the mod construction, a smaller-than-average shard would overshoot — `interleave_datasets`'s default `first_exhausted` strategy then collapses the whole eval round to zero batches.

In [ ]:
fails = 0
checked = 0
for repo_id, name in SOURCES:
    for s in range(256):
        pick = eval_shard_pick.pick_shard_for_source(repo_id=repo_id, name=name, int_seed=s)
        if not (0 <= pick.in_shard_offset < pick.offset_bound):
            fails += 1
            print(f"  FAIL seed={s} shard={pick.shard_path}"
                  f" offset={pick.in_shard_offset} bound={pick.offset_bound}")
        checked += 1
print(f"{checked} picks checked, {fails} violations")

## (3) Per-source independence

For each seed, source A's pick and source B's pick should be independent draws. If they correlate (e.g. both always at shard index ≡ seed mod N), a miner who guesses one learns the other for free.

Skip this cell if only one source is enabled above.

In [ ]:
if len(SOURCES) >= 2:
    coincidences = 0
    N = 64
    for s in range(N):
        picks = [eval_shard_pick.pick_shard_for_source(repo_id=r, name=n, int_seed=s) for r, n in SOURCES]
        idxs = [int(p.shard_path.split('.')[1].split('-')[0]) if '.json.gz' in p.shard_path
                else int(p.shard_path.split('_')[-1].split('.')[0])
                for p in picks]
        if all(i == idxs[0] for i in idxs):
            coincidences += 1
    print(f"all-same shard-index across sources: {coincidences}/{N} seeds")
    print("(expected ~0 if hash domain separation is working)")
else:
    print("single source configured — skipping independence check")

## (4) Cross-seed coverage

Across 4096 seeds, every shard should be picked roughly `4096 / shard_count` times. For C4-en (1024 shards) that's ~4 picks per shard on average. A real bug — say, the hash collapses to a small subset — would show up as one shard taking 100s of picks while others get 0.

In [ ]:
for repo_id, name in SOURCES:
    counts: Counter[str] = Counter()
    for s in range(4096):
        pick = eval_shard_pick.pick_shard_for_source(repo_id=repo_id, name=name, int_seed=s)
        counts[pick.shard_path] += 1
    distinct = len(counts)
    max_picks = max(counts.values())
    min_picks = min(counts.values()) if counts else 0
    mean = sum(counts.values()) / max(1, distinct)
    print(f"{repo_id:35s} distinct_shards={distinct:>5d} mean={mean:6.2f} min={min_picks:>3d} max={max_picks:>3d}")

## (5) Live shard read — eyeball the sample

This is the production path: pick a shard, open a streaming dataset on it, skip by the offset, pull a few rows. If this cell prints intelligible text from somewhere in the *middle* of the C4 dataset (not always from row 0 of shard 0), the new path works end to end.

Sets `MY_INT_SEED` from above. Re-run with a different seed to confirm each seed lands at a different spot.

Network/disk cost: one HTTP fetch (~320 MB compressed for C4-en) + parse + skip. Cached after first run if `HF_HUB_CACHE` is writable.

In [ ]:
import itertools

for repo_id, name in SOURCES:
    pick = eval_shard_pick.pick_shard_for_source(repo_id=repo_id, name=name, int_seed=MY_INT_SEED)
    print(f"\n--- {repo_id} ({name}) ---")
    print(f"shard:  {pick.shard_path}")
    print(f"offset: {pick.in_shard_offset:,} / {pick.shard_rows:,}")
    print("opening stream + applying skip ...")
    ds = eval_shard_pick.load_streaming_shard(pick)
    ds = ds.skip(pick.in_shard_offset)
    rows = list(itertools.islice(ds, 3))
    for i, row in enumerate(rows):
        # Each source has its own column layout; the 'text' column is the validator's input.
        text = row.get("text") or next(iter(row.values()))
        print(f"\n  [row {i}] {text[:200]!r}" + ("..." if len(text) > 200 else ""))

## (6) Equivalence check — do `data_files=` rows match a canonical read?

`data_files=[shard]` bypasses any HF dataset loading script. For C4 this is fine (raw json.gz, no script-side processing), but it's worth verifying that the rows yielded match what a canonical `load_dataset(repo_id, name=name, streaming=True)` call would yield at the same depth.

This cell does a tiny spot check: shard 0 row 0 via the new path should equal shard 0 row 0 via the canonical path. If they differ in any of the columns we care about (`text` in particular), the `data_files=` path is doing something we don't want and the gate should not be flipped until that's resolved.

Only runs for sources where the canonical load path doesn't require auth (i.e. C4).

In [ ]:
from datasets import load_dataset
import itertools

# Skip this cell if a pandas/numpy ABI mismatch is in your local env;
# the check is best-effort. The unit tests cover the algebraic
# properties; this is the live equivalence read.
try:
    canon = load_dataset("allenai/c4", "en", streaming=True)["train"]
    first_canon = next(iter(canon))
    print("canonical row 0 (first 120 chars):", first_canon["text"][:120], "...")

    pick = eval_shard_pick.ShardPick(
        repo_id="allenai/c4",
        name="en",
        revision="main",
        shard_path="en/c4-train.00000-of-01024.json.gz",
        offset_bound=340_000,
        shard_rows=340_000,  # for the constant path this mirrors offset_bound
        in_shard_offset=0,
    )
    via_pick = eval_shard_pick.load_streaming_shard(pick)
    first_pick = next(iter(via_pick))
    print("shard-pick row 0 (first 120 chars):", first_pick["text"][:120], "...")

    match = first_canon["text"] == first_pick["text"]
    print(f"\n  TEXTS MATCH = {match}")
    if not match:
        print("  -> DO NOT flip the gate. Investigate the data_files= load path.")
except Exception as e:
    print(f"equivalence check skipped: {e!r}")

## What to do before flipping `eval_source_seeded_shard_pick=True`

1. **Re-verify shard sizes against the current dataset revision.** For C4, HEAD-fetch a handful of shards (say 5-10 spread across the index range) and confirm row counts are consistent with `_KNOWN_SOURCES["allenai/c4", "en"].verified_shard_rows`. If a shard's actual size has dropped below `safe_floor_rows + min_headroom_rows`, lower the floor (or, in extreme cases, switch the source off shard-pick entirely).

2. **Pin per-source HF commit SHAs** via `DataCfg.eval_source_revision_pin`. `"main"` is a moving target; an upstream re-upload during rollout would cause two validators to pick different rows for the same seed and break weight consensus.

3. **Run cell (6) for every configured source** including Nemotron (requires HF auth). If `data_files=[shard]` yields rows that differ from the canonical loader, do NOT flip the gate.

4. **Coordinate the rollout at a chain epoch**, same as the original `eval_source_shuffle_buffer` flip. Validators on different gate values will produce non-comparable losses for the transition round and consensus will break for ~1 round.

## What changed from path A

Path A (replaced) maintained a full per-shard row-count table in `connito/shared/data/c4_en_shard_rows.json` for exact mod bounds, requiring an offline enumeration script to populate ~1024 entries.

Path B (current) replaces that table with a small `verified_shard_rows` spot-check dict baked into the policy registry. The mod bound becomes a per-source `safe_floor_rows` constant for json.gz sources, or the live parquet footer for parquet sources. Module-load validation ensures `safe_floor + min_headroom ≤ min(verified)` so the read window can never run short. Trade-off: ~4.6% per-shard reach loss for C4 in exchange for no offline enumeration pre-rollout.